In [2]:
import sys
import subprocess
import importlib

In [3]:
def ensure(pkg, import_as=None):
    try:
        importlib.import_module(import_as or pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            pkg
        ])

packages = [
    ("ultralytics", "ultralytics"),
    ("opencv-python", "cv2"),
    ("numpy", "numpy"),
    ("torch", "torch"),
    ("torchvision", "torchvision"),
    ("scipy", "scipy"),
]

for p, i in packages:
    ensure(p, i)


In [4]:
import cv2
import math
import time
import torch
import numpy as np
from ultralytics import YOLO
from scipy.ndimage import gaussian_filter
from collections import deque


In [5]:
GREEN = (0,255,0)
RED = (0,0,255)
YELLOW = (0,255,255)
BLUE = (255,0,0)
WHITE = (255,255,255)
ORANGE = (0,165,255)

In [6]:
# SETTINGS

In [7]:
PERSON_CLASS = 0
PANIC_SPEED = 18
TRAIL = 25

In [8]:
# TRACK CLASS

In [9]:
class PersonTrack:

    def __init__(self, tid, bbox):
        self.id = tid
        self.bbox = bbox
        self.history = deque(maxlen=TRAIL)
        self.velocities = deque(maxlen=10)

        cx, cy = self.center(bbox)
        self.history.append((cx, cy))

    def center(self, b):
        x1, y1, x2, y2 = b
        return int((x1+x2)/2), int((y1+y2)/2)

    def update(self, bbox):
        px, py = self.history[-1]
        cx, cy = self.center(bbox)

        dx = cx - px
        dy = cy - py

        speed = math.hypot(dx, dy)

        self.velocities.append(speed)
        self.history.append((cx, cy))
        self.bbox = bbox

    @property
    def avg_speed(self):
        if len(self.velocities) == 0:
            return 0
        return np.mean(self.velocities)


In [10]:
# SIMPLE TRACKER

In [11]:
class Tracker:

    def __init__(self):
        self.tracks = {}
        self.next_id = 1

    def distance(self, a, b):
        ax1, ay1, ax2, ay2 = a
        bx1, by1, bx2, by2 = b

        acx = (ax1+ax2)/2
        acy = (ay1+ay2)/2

        bcx = (bx1+bx2)/2
        bcy = (by1+by2)/2

        return math.hypot(acx-bcx, acy-bcy)

    def update(self, detections):

        updated = {}

        for det in detections:
            matched = False

            for tid, tr in self.tracks.items():
                d = self.distance(tr.bbox, det)

                if d < 50:
                    tr.update(det)
                    updated[tid] = tr
                    matched = True
                    break

            if not matched:
                tr = PersonTrack(self.next_id, det)
                updated[self.next_id] = tr
                self.next_id += 1

        self.tracks = updated
        return self.tracks


In [12]:
# OPTICAL FLOW

In [13]:
class FlowEngine:

    def __init__(self):
        self.prev = None
        self.flow = None

    def update(self, frame):

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        if self.prev is None:
            self.prev = gray
            return

        self.flow = cv2.calcOpticalFlowFarneback(
            self.prev,
            gray,
            None,
            0.5,
            3,
            15,
            3,
            5,
            1.2,
            0
        )

        self.prev = gray

    def entropy(self):

        if self.flow is None:
            return 0

        dx = self.flow[:,:,0]
        dy = self.flow[:,:,1]

        mag = np.hypot(dx, dy)

        return np.clip(np.mean(mag)/10, 0, 1)


In [14]:
# HEATMAP

In [15]:
class HeatMap:

    def __init__(self, h, w):
        self.map = np.zeros((h,w), dtype=np.float32)

    def update(self, tracks):

        for t in tracks.values():
            cx, cy = t.history[-1]

            if 0 <= cy < self.map.shape[0] and 0 <= cx < self.map.shape[1]:
                self.map[cy, cx] += 1

        self.map *= 0.97

    def render(self, frame):

        blur = gaussian_filter(self.map, sigma=20)

        mx = blur.max()

        if mx < 1:
            return

        norm = (blur/mx*255).astype(np.uint8)

        color = cv2.applyColorMap(norm, cv2.COLORMAP_JET)

        cv2.addWeighted(color, 0.35, frame, 0.65, 0, frame)


In [16]:
# PANIC ANALYZER

In [17]:
class PanicAnalyzer:

    def analyze(self, tracks, entropy, crowd_count):

        speeds = [t.avg_speed for t in tracks.values()]

        if len(speeds) == 0:
            return 0, "NORMAL"

        speed_spike = sum(s > PANIC_SPEED for s in speeds) / len(speeds)

        density = min(crowd_count / 40, 1)

        panic_score = (
            speed_spike * 0.5 +
            entropy * 0.3 +
            density * 0.2
        )

        panic_score = np.clip(panic_score, 0, 1)

        if panic_score < 0.3:
            state = "NORMAL"
        elif panic_score < 0.55:
            state = "ALERT"
        elif panic_score < 0.75:
            state = "PANIC RISK"
        else:
            state = "MASS PANIC"

        return panic_score, state


In [18]:
# DRAW HELPERS

In [19]:
def draw_text(frame, text, x, y, color=WHITE):
    cv2.putText(
        frame,
        text,
        (x,y),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        color,
        2,
        cv2.LINE_AA
    )

In [20]:
# MAIN SYSTEM

In [ ]:
class CrowdPanicAI:

    def __init__(self, source=0):

        self.device = "cuda" if torch.cuda.is_available() else "cpu"

        print("Loading YOLOv8...")

        self.model = YOLO("yolov8n.pt")

        self.cap = cv2.VideoCapture(source)

        self.W = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.H = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

        self.tracker = Tracker()
        self.flow = FlowEngine()
        self.analyzer = PanicAnalyzer()
        self.heatmap = HeatMap(self.H, self.W)

        self.writer = cv2.VideoWriter(
            "output/output.mp4",
            cv2.VideoWriter_fourcc(*"mp4v"),
            30,
            (self.W, self.H)
        )

    def detect_people(self, frame):

        results = self.model(frame, classes=[0], verbose=False)

        boxes = []

        for r in results:

            if r.boxes is None:
                continue

            for b in r.boxes:
                x1, y1, x2, y2 = map(int, b.xyxy[0])
                boxes.append((x1,y1,x2,y2))

        return boxes

    def draw_tracks(self, frame, tracks, panic_score):

        for t in tracks.values():

            x1, y1, x2, y2 = t.bbox

            speed = t.avg_speed

            if speed > PANIC_SPEED:
                color = RED
            elif panic_score > 0.5:
                color = ORANGE
            else:
                color = GREEN

            cv2.rectangle(frame, (x1,y1), (x2,y2), color, 2)

            draw_text(
                frame,
                f"ID:{t.id} {speed:.1f}",
                x1,
                y1-10,
                color
            )

            pts = list(t.history)

            for i in range(1, len(pts)):
                cv2.line(frame, pts[i-1], pts[i], color, 2)
    
    def draw_dashboard(self, frame, panic_score, state, count, fps):

        h, w = frame.shape[:2]

        cv2.rectangle(frame, (0,0), (350,220), (20,20,20), -1)

        draw_text(frame, "SMART CROWD PANIC AI", 20, 30, BLUE)

        draw_text(frame, f"FPS: {fps:.1f}", 20, 60, GREEN)
        draw_text(frame, f"People: {count}", 20, 90, YELLOW)
        draw_text(frame, f"Panic Score: {panic_score:.2f}", 20, 120, ORANGE)
        draw_text(frame, f"State: {state}", 20, 150, RED)

        bar = int(panic_score * 300)

        cv2.rectangle(frame, (20,180), (320,200), WHITE, 2)
        cv2.rectangle(frame, (20,180), (20+bar,200), RED, -1)

        if state == "MASS PANIC":
            draw_text(frame, "EMERGENCY ALERT !!!", w//2-150, 50, RED)

    
    def run(self):

        prev = time.time()

        while True:

            ret, frame = self.cap.read()

            if not ret:
                break

            
            # DETECT
            

            detections = self.detect_people(frame)

                
            # TRACK
            

            tracks = self.tracker.update(detections)

            
            # FLOW
            
            self.flow.update(frame)

            entropy = self.flow.entropy()

            
            # PANIC ANALYSIS
            

            panic_score, state = self.analyzer.analyze(
                tracks,
                entropy,
                len(tracks)
            )

            
            # HEATMAP
            

            self.heatmap.update(tracks)
            self.heatmap.render(frame)
            
            
            # DRAW TRACKS
            

            self.draw_tracks(frame, tracks, panic_score)

            
            # FPS
            

            now = time.time()
            fps = 1/(now-prev)
            prev = now

            
            # DASHBOARD
            

            self.draw_dashboard(
                frame,
                panic_score,
                state,
                len(tracks),
                fps
            )

            self.writer.write(frame)

            cv2.imshow("Crowd Panic AI", frame)

            key = cv2.waitKey(1)

            if key == ord('q'):
                break

            if key == ord('s'):
                cv2.imwrite(
                    f"screenshots/frame_{time.time()}.png",
                    frame
                )

        self.cap.release()
        self.writer.release()
        cv2.destroyAllWindows()

In [22]:
if __name__ == "__main__":

    print("SMART CROWD PANIC DETECTION AI")

    source = r"D:\Projects\Smart Crowd Panic Detection\videos\YTDown_YouTube_People-Walking-Free-Stock-Footage-Royalt_Media_YzcawvDGe4Y_002_720p.mp4"
    

    ai = CrowdPanicAI(source)

    ai.run()

SMART CROWD PANIC DETECTION AI
Loading YOLOv8...
